In [3]:
import pandas as pd
import numpy as np

# Import clean data
path = 'https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DA0101EN-SkillsNetwork/labs/Data%20files/module_5_auto.csv'
df = pd.read_csv(path)

df.to_csv('module_5_auto.csv')


# _get_numeric_data() 是 pandas DataFrame 的一个方法，返回一个只包含数值型列（int、float 等）的新 DataFrame，字符串列（如 make、fuel-type、body-style）、布尔列等都会被自动剔除。
# df = df._get_numeric_data() 用筛选结果覆盖原来的 df，后续所有操作都基于这个纯数值的数据集。
df=df._get_numeric_data()
df.head()


import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

# DistributionPlot，用于对比两条数据的概率密度分布曲线（典型用法：真实价格 vs 模型预测价格的分布对比）
# RedFunction：第一条要绘制的数据（通常画成红色，本 lab 中常是真实值，如 y_train 或 y_test）
# BlueFunction：第二条要绘制的数据（蓝色，通常是模型预测值，如 yhat_train）
# RedName / BlueName：两条曲线的图例标签文字
# Title：图表标题
# 用 seaborn 的 kdeplot（核密度估计图，Kernel Density Estimate）绘制第一组数据：
# DE 是对数据分布的平滑连续估计，相当于平滑版的直方图，能直观看出价格的分布形状（峰值位置、宽度、偏态）
# color="r" 红色；label=RedName 设置图例文字
# 返回值 ax1 是 matplotlib 的坐标轴对象，供下一条曲线共用
# 关键在 ax=ax1：显式指定画到 ax1 上。如果不传这个参数，seaborn 可能自动新建画布，两条曲线就分开了。这样两条曲线叠在一起，才能直接对比。
def DistributionPlot(RedFunction, BlueFunction, RedName, BlueName, Title):
    width = 12
    height = 10
    plt.figure(figsize=(width, height))

    ax1 = sns.kdeplot(RedFunction, color="r", label=RedName)
    ax2 = sns.kdeplot(BlueFunction, color="b", label=BlueName, ax=ax1)

    plt.title(Title)
    plt.xlabel('Price (in dollars)')
    plt.ylabel('Proportion of Cars')
    plt.show()
    plt.close()

# xtrain / xtest：训练集/测试集的特征值（本 lab 中是 horsepower 这一列）
# y_train / y_test：对应的真实价格
# lr：已训练好的线性回归模型对象（如 poly = LinearRegression()）
# poly_transform：多项式特征变换器（如 pr = PolynomialFeatures(degree=5)）
# 第二个可视化辅助函数 PollyPlot，用于把训练数据点、测试数据点和多项式回归模型的预测曲线画在同一张图上，直观展示模型的拟合效果
# 取训练集和测试集中特征的最大值/最小值，确定横轴要覆盖的完整范围——这样画出的预测曲线能同时覆盖两类数据点的区域。
# 绘制训练数据：'ro' 是格式字符串——r = 红色，o = 圆点标记。即红色圆点 (x=马力， y=价格)，图例标注 "Training Data"。
# 绘制测试数据：'go' = 绿色圆点，图例 "Test Data"。红/绿两色区分了模型“见过”和“没见过”的数据

def PollyPlot(xtrain, xtest, y_train, y_test, lr,poly_transform):
    width = 12
    height = 10
    plt.figure(figsize=(width, height))


    #training data
    #testing data
    # lr:  linear regression object
    #poly_transform:  polynomial transformation object


    #骨架
    xmax=max([xtrain.values.max(), xtest.values.max()])

    xmin=min([xtrain.values.min(), xtest.values.min()])

    x=np.arange(xmin, xmax, 0.1)


    plt.plot(xtrain, y_train, 'ro', label='Training Data')
    plt.plot(xtest, y_test, 'go', label='Test Data')
    # x.reshape(-1, 1)：把一维数组 x 变成形状 (n, 1) 的二维列向量——sklearn 要求特征输入是二维的
    # 变换前: [50.0, 50.1, 50.2, ...]        形状 (n,)     一行数字
    # 变换后: [[50.0], [50.1], [50.2], ...]  形状 (n, 1)   n行×1列
    # poly_transform.fit_transform(...)：对网格点做多项式特征展开
    # poly_transform.fit_transform(x.reshape(-1, 1))
    # (n, 1) → (n, 6)   [1, x, x², x³, x⁴, x⁵]  把网格点升维，这个维度不一定，要看你函数外传入的值
    # lr.predict(...)
    #  对网格里的每个马力值，用学到的方程算出预测价格
    #  输出形状 (n,)：[预测价格₁, 预测价格₂, ...]
    # 如 degree=5 时生成 [x, x², x³, x⁴, x⁵]），必须与训练模型时用的变换器参数（degree）一致，否则维度不匹配
    # 用训练好的模型对每个网格点预测价格
    # plt.plot(x, ...)：把“马力 → 预测价格”画成一条线（默认蓝色实线），即模型的预测函数曲线
    plt.plot(x, lr.predict(poly_transform.fit_transform(x.reshape(-1, 1))), label='Predicted Function')
    # 固定纵轴范围为 -10000 到 60000 美元。这一步很关键：degree 高的多项式在数据边界外会剧烈震荡甚至发散（例如 200+ 马力区域预测值飞出几千甚至几万美元），固定 y 轴范围能防止图形被极端值撑爆，让我们清楚看到曲线“失控”的样子。
    plt.ylim([-10000, 60000])
    plt.ylabel('Price')
    # plt.legend() 的作用是显示图例
    plt.legend()


# 训练集与测试集
#要预测的车价
y_data = df['price']
# 除了车价中的所有列
x_data=df.drop('price',axis=1)

# 这是 Part 1（训练与测试）中数据集划分的核心代码。
# 从 sklearn 的 model_selection 模块导入 train_test_split 函数——这是 sklearn 官方提供的、把数据集随机拆分为训练集/测试集的标准工具。
# x_data, y_data：待划分的特征和目标。函数内部会按同样的随机顺序打乱两者的行，保证特征与标签的对应关系不被破坏
# test_size=0.10：10% 的样本划入测试集，其余 90% 划入训练集
# random_state=1：随机种子。固定种子后，每次运行得到的划分结果完全一致（可复现）。这是教学/作业中的关键设置——否则每次运行 notebook 的指标都会变，无法对齐参考答案
# 变量	        内容	        行数
# x_train	训练集特征	    90%
# x_test	测试集特征	    10%
# y_train	训练集目标（价格）	90%
# y_test	测试集目标（价格）	10%

 # 易错点：x_train 必须配 y_train，x_test 必须配 y_test，写错配对会导致模型学到错误对应关系。
#  .shape 返回 (行数, 列数)，[0] 取行数即样本数
# 用来验证划分比例：汽车数据集共 201 行，10% 测试 → 输出大约 test: 21、training: 180（sklearn 用 ceil 计算测试集数量，所以是 21 而非 20）

from sklearn.model_selection import train_test_split

#专门划分训练集和测试集用的
x_train, x_test, y_train, y_test = train_test_split(x_data, y_data, test_size=0.10, random_state=1)


print("number of test samples :", x_test.shape[0])
print("number of training samples:",x_train.shape[0])


# 为什么要划分？
# 训练集用来让模型“学习”规律，测试集（模型从未见过的数据）用来模拟真实场景下的表现。如果直接在训练数据上评估，模型可能只是背下了答案（过拟合），分数虚高；
# 第 1 步：划分数据（10% 留作测试）
# 第 2 步：用 90% 的训练集"喂"给模型学习 → 得到模型
# 第 3 步：把模型放进测试集"考试"

